In [1]:
import xarray as xr
from pystac_client import Client
import lazycogs
import stac_geoparquet
import geopandas as gpd
import pandas as pd
import obstore
import re
import numpy as np
import rasterio
import xvec
from functools import reduce
import itertools
import rioxarray
import shapely

In [2]:
csde_stac = Client.open("https://stac.dataspace.copernicus.eu/v1/")
lsp_items = csde_stac.search(
    collections="clms_lsp_global_300m_yearly_v2_cog"
).item_collection()

In [3]:
lsp_items_dict = lsp_items.to_dict()

In [4]:
lsp_items_dict["features"][0]

{'type': 'Feature',
 'stac_version': '1.1.0',
 'stac_extensions': ['https://cs-si.github.io/eopf-stac-extension/v1.2.0/schema.json',
  'https://stac-extensions.github.io/alternate-assets/v1.2.0/schema.json',
  'https://stac-extensions.github.io/authentication/v1.1.0/schema.json',
  'https://stac-extensions.github.io/file/v2.1.0/schema.json',
  'https://stac-extensions.github.io/processing/v1.2.0/schema.json',
  'https://stac-extensions.github.io/product/v1.0.0/schema.json',
  'https://stac-extensions.github.io/projection/v2.0.0/schema.json',
  'https://stac-extensions.github.io/storage/v2.0.0/schema.json',
  'https://stac-extensions.github.io/timestamps/v1.1.0/schema.json'],
 'id': 'c_gls_LSP300-TPROD-S2_202501010000_GLOBE_OLCI_V2.0.1_cog',
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-179.9999999, 80.0014881],
    [-179.9999999, -59.998513],
    [-0.0007425999999953, -59.998513],
    [179.9985148, -59.998513],
    [179.9985148, 80.0014881],
    [-0.0007425999999953, 80.0014881

In [5]:
target_assets = [
    "EOSD-S1",
    "EOSD-S2",
    "SOSD-S1",
    "SOSD-S2",
    "AMPL-S1",
    "AMPL-S2",
    "QA-S1",
    "QA-S2"
]

def sosd_eosd_filter(item: dict) -> bool:
    return any(asset in item["id"] for asset in target_assets)

eosd_sosd_items = list(filter(sosd_eosd_filter, lsp_items_dict["features"]))

In [6]:
len(eosd_sosd_items)

96

In [7]:
for item in eosd_sosd_items:
    # Remove property keys that are not always present and cause failure on conversion
    # to geoparquet.
    item["properties"].pop("platform", None)
    item["properties"].pop("constellation", None)
    # Remove incorrect datetime properties
    item["properties"].pop("datetime", None)
    item["properties"].pop("start_datetime", None)
    item["properties"].pop("end_datetime", None)
    # Parse the correct datetime from the item ID
    dt_str = item["id"].split("_")[3]
    item["properties"]["datetime"] = pd.to_datetime(dt_str, utc=True)
    # Add asset-level proj code so lazycogs knows what's going on
    for asset_key in item["assets"]:
        item["assets"][asset_key]["proj:code"] = "EPSG:4326"

In [8]:
stac_geoparquet.to_geodataframe(eosd_sosd_items, dtype_backend="numpy_nullable").to_parquet("../data_working/stac_cache/lsp_items.parquet")

In [9]:
print(list(reduce(set.union, (set(i["assets"].keys()) for i in eosd_sosd_items))))

['lsp300_eosd_s1', 'lsp300_eosd_s2', 'lsp300_ampl_s1', 'thumbnail', 'lsp300_qa_s1', 'Product', 'lsp300_ampl_s2', 'lsp300_qa_s2', 'lsp300_sosd_s2', 'lsp300_sosd_s1']


In [10]:
parquet_reopen = gpd.read_parquet("../data_working/stac_cache/lsp_items.parquet")

In [11]:
parquet_reopen.columns

Index(['type', 'stac_version', 'stac_extensions', 'id', 'geometry', 'bbox',
       'links', 'assets', 'collection', 'gsd', 'created', 'expires', 'updated',
       '_private', 'proj:code', 'published', 'instruments', 'auth:schemes',
       'product:type', 'storage:schemes', 'processing:level',
       'processing:version', 'product:timeliness', 'processing:software',
       'eopf:origin_datetime', 'product:timeliness_category', 'datetime'],
      dtype='str')

Set up S3 credentials

In [12]:
import configparser

config = configparser.ConfigParser()
config.read("/home/jovyan/.s3cfg")

access_key = config.get("cdse", "access_key")
secret_key = config.get("cdse", "secret_key")

store = obstore.store.S3Store(
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    bucket="eodata",
    endpoint="https://eodata.dataspace.copernicus.eu"
)

In [13]:
all_geometries = gpd.read_file("../data_working/deadtrees_aoi.gpkg").set_index("dataset_id")
# Toy geometries
all_geometries.explore()

In [14]:
target_bands = [
    "lsp300_" + asset.lower().replace("-", "_")
    for asset in target_assets
]

In [15]:
def zonal_stats_driver(geom: gpd.GeoDataFrame, da: xr.DataArray):
    # Clip box ahead of zonal_stats means that we don't have to rasterize
    # geometry over the entire data extent.
    da = da.rio.clip_box(*shapely.bounds(geom.geometry.union_all().buffer(300)), allow_one_dimensional_raster=True).load()
    da = da.where(da != -9999).squeeze()
    
    zs = da.xvec.zonal_stats(geom.geometry, "x", "y", all_touched=True)\
        .assign_coords(geometry=geom.index.rename("geometry"))\
        .to_pandas()

    zs_melt = pd.melt(zs, ignore_index=False, value_name=str(da.band.data))
    return zs_melt

In [16]:
def zonal_stats_on_asset(geometry: gpd.GeoDataFrame, asset_name: str, band_name: str) -> pd.DataFrame:
    # Open array..
    target_ids = list(filter(
        lambda x: asset_name in x,
        (i["id"] for i in eosd_sosd_items)
    ))

    lsp_da = lazycogs.open(
        "../data_working/stac_cache/lsp_items.parquet",
        bands=[band_name],
        ids=target_ids,
        store=store,
        crs="EPSG:8857",
        resolution=300,
        max_concurrent_reads=4,
        bbox=geometry.total_bounds
    )
    

    # Run zonal statistics. This we have to run on each subtree because
    # geometry rasterization is memory intensive.
    this_zs = geometry.groupby("subtree", group_keys=False).apply(zonal_stats_driver, lsp_da)

    return this_zs

In [17]:
%%time
import itertools
zs_dataframes = []
for (asset, band) in zip(target_assets, target_bands):
    print(f"Starting {asset}, {band}")
    this_df = zonal_stats_on_asset(all_geometries, asset, band)
    zs_dataframes.append(this_df)

Starting EOSD-S1, lsp300_eosd_s1
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting EOSD-S2, lsp300_eosd_s2
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting SOSD-S1, lsp300_sosd_s1
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting SOSD-S2, lsp300_sosd_s2
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting AMPL-S1, lsp300_ampl_s1
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting AMPL-S2, lsp300_ampl_s2
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting QA-S1, lsp300_qa_s1
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
Starting QA-S2, lsp300_qa_s2
INFO:lazycogs._core:Discovered 1 bands and 12 time steps.
CPU times: user 39min 17s, sys: 5min 25s, total: 44min 43s
Wall time: 2h 12min 6s


In [18]:
# Promote time to an index
zs_dataframes = [df.set_index("time", append=True) for df in zs_dataframes]

In [19]:
all_zs = pd.concat(zs_dataframes, axis=1, join="inner")

In [20]:
all_zs.to_parquet("../data_working/deadtrees/aoi_growing_season.parquet")